In [0]:
with open("/dbfs/databricks-datasets/tpch/README.md", "r") as f:
    content = f.read()
    displayHTML(f"<pre>{content}</pre>")

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bda_silver")

## Validation rules

Non-empty brand and manufacturer

In [0]:
from pyspark.sql.functions import col, trim, when, lit

df_part_bronze = spark.read.table("bda_bronze.part")

# Invalid records → quarantine
df_part_quarantine = df_part_bronze.filter(
    col("p_brand").isNull() |
    (trim(col("p_brand")) == "") |
    col("p_mfgr").isNull() |
    (trim(col("p_mfgr")) == "")
).withColumn(
    "rejection_reason",
    when(
        col("p_brand").isNull() | (trim(col("p_brand")) == ""),
        lit("Empty or NULL brand")
    ).when(
        col("p_mfgr").isNull() | (trim(col("p_mfgr")) == ""),
        lit("Empty or NULL manufacturer")
    )
)
df_part_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bda_silver.part_quarantine")

# Valid records → Silver
df_part_silver = df_part_bronze.filter(
    col("p_brand").isNotNull() &
    (trim(col("p_brand")) != "") &
    col("p_mfgr").isNotNull() &
    (trim(col("p_mfgr")) != "")
)

df_part_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bda_silver.part")

# Enforce constraints on Silver
spark.sql("""
ALTER TABLE bda_silver.part
ALTER COLUMN p_brand SET NOT NULL
""")

spark.sql("""
ALTER TABLE bda_silver.part
ALTER COLUMN p_mfgr SET NOT NULL
""")

spark.sql("""
ALTER TABLE bda_silver.part
ADD CONSTRAINT valid_brand
CHECK (TRIM(p_brand) <> '')
""")

spark.sql("""
ALTER TABLE bda_silver.part
ADD CONSTRAINT valid_manufacturer
CHECK (TRIM(p_mfgr) <> '')
""")

In [0]:
df_lineitem_bronze = spark.read.table("bda_bronze.lineitem")
df_partsupp_bronze = spark.read.table("bda_bronze.partsupp").select(
    "ps_partkey", "ps_suppkey", "ps_supplycost"
)

df_lineitem_silver = (
    df_lineitem_bronze.join(
        df_partsupp_bronze,
        (col("l_partkey") == col("ps_partkey")) &
        (col("l_suppkey") == col("ps_suppkey")),
        "left"
    )
    .withColumn(
        "revenue",
        col("l_extendedprice") * (1 - col("l_discount"))
    )
    .withColumn(
        "margin",
        col("revenue") - col("l_quantity") * col("ps_supplycost")
    )
    .drop("ps_partkey", "ps_suppkey", "ps_supplycost")
)
df_lineitem_silver.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bda_silver.lineitem")

In [0]:
# NOTE: "part" is excluded — it was already written with validation
# and constraints above.  Overwriting it here would bypass the
# quarantine-based quality flow.
remaining_tables = [
    "customer", "nation", "orders",
    "partsupp", "region", "supplier"
]

for table in remaining_tables:
    df = spark.read.table(f"bda_bronze.{table}")
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"bda_silver.{table}")